# 5 · Evaluation — scoring the arms, the external benchmark, and diagnostics

> **Corrected 13.08.2026** (Selin, review item 11). Until today this file was a stub whose stated
> remit was *"external benchmark and diagnostics"*, held empty because neither source notebook could
> run. Both halves of that have changed. The remit was too narrow — `58fadd7` had already assigned
> this stage the scoring of the training arms — and the blockers it listed are fixed. What that stub
> said is preserved below under [What was believed on 12.08.2026](#what-was-believed-on-12082026),
> because it was true when written.

## What this stage does

Three sections, and they do **not** all run at the same point of the sweep. That is deliberate, but it
means running this notebook top to bottom before R5 will fail halfway — by design, not by breakage.

| § | what it does | runs at | why not earlier |
|---|---|---|---|
| **1 · Score the arms** | the four quantities below, over the out-of-fold predictions `4a_percell_training` writes | **R4** | needs `panel_oof_predictions.csv` and nothing else. It is needed *at* R4 rather than after, because the loss comparison (item 9A) cannot be judged without the calibration slope in §1. |
| **2 · External benchmark** | absorbs [`analysis/evaluation/dreval_benchmark.ipynb`](analysis/evaluation/dreval_benchmark.ipynb) — OncoMLP against DrEval's own baselines, their splits, their metrics | **R5** | consumes retrained outputs and the `auc_cc` targets file, neither of which exists before the sweep runs. |
| **3 · Diagnostics** | absorbs [`analysis/evaluation/diagnostics.ipynb`](analysis/evaluation/diagnostics.ipynb) — proliferation test, input scale, result dispersion | **R5** | same. |

Sections 2 and 3 stay in `analysis/evaluation/` until they run; they move here when they do. Moving a
notebook that raises on its first cell into a numbered stage would put a broken step in the middle of a
chain that is meant to be a path you can walk.

## The four quantities

Fixed by Selin in `58fadd7`, so that the per-cell and MIL architectures are *"comparable by
construction rather than by convention"* — §1 computes them through identical code for both.

| quantity | the question it answers |
|---|---|
| **order** | within a drug, does the predicted ranking of cell lines match the true one? |
| **top-of-order** | are the cell lines called most extreme actually the most extreme? |
| **values** | how far off are the predictions, against a per-drug constant? |
| **spread** | does the model use the real range, or collapse toward the mean? |

**Spread is measured as a calibration slope with its intercept reported alongside** (Selin,
13.08.2026), not as a ratio of standard deviations. The pair is the point: the slope catches
compression, the intercept catches shift, and a model can be flat *and* shifted. Van Calster, Nieboer,
Vickers, Van Calster & Steyerberg, *A calibration hierarchy for risk models*, **J Clin Epidemiol 74
(2016) 167–176**.

## What is not decided yet

These are open and belong to Selin. Each is marked again at the cell that needs it, and no cell
resolves one by being written.

- **Which quantity is primary** in the item-9A loss comparison. The *shape* is decided —
  one quantity decides, the other three act as non-inferiority guards — and so is the margin,
  **±0.04**, which is the recorded seed band from item 9A rather than a chosen number, over **≥3
  seeds**. Which of the four sits in the primary slot is not. §1 takes it as a named parameter.
- **The second baseline** in §1: whether to score against a per-drug constant alone, or also against a
  per-cell-line mean across drugs.
- **Squared or absolute error** for *values*.
- **Folds or seeds** for the dispersion shown in the summary table.

## What was believed on 12.08.2026

The stub this replaces gave two reasons the stage could not be written, and recorded them in a
*"why it cannot run today"* column. Both were accurate then and neither holds now:

- *"`dreval_benchmark` hardcodes `'auc'`, a score `layout.CTRP_SCORES` has rejected since 11.08.2026,
  so `PipelinePaths.build` raises on construction."* — fixed for both notebooks in `e804f07`; they
  read `DEFAULT_CTRP_SCORE`.
- *"It imports the cell-line-effect diagnostic that was deleted on 12.08.2026."* — rewired to DrEval's
  own recipe in `af2cfa9`. That diagnostic stays retired.

Two things the stub flagged **do** still hold, and are not closed by the above:

1. Under leave-cell-line-out, DrEval's normalization removes the **drug** effect only — a held-out
   line's effect is unseen and therefore zero. A synthetic predictor emitting nothing but
   `mean + line effect + drug effect` scores normalized Spearman **0.98**. So *"drug-specific signal,
   or general cell-line fragility?"* is a real question their metric does not answer under our split
   design, and answering it needs a diagnostic that reads held-out labels rather than a metric.
   Whether such a diagnostic returns, and in what form, is still item 11's to settle
   ([`scripts/archive/README.md`](../scripts/archive/README.md)).
2. `dreval_normalize.py` requires a `fold` column so the naive baseline is fitted on the folds a
   prediction did *not* come from. The committed
   `outputs/legacy/panel_void_8drug/panel_oof_predictions.csv` predates that column and the script
   raises on it, correctly. It becomes runnable when [stage 4a](4a_percell_training.ipynb) re-runs.

> ⛔ Nothing is re-run until the 03.08.2026 freeze in [TODO](../docs/TODO.md) lifts.